# Frontieres-03 — MAPF : auditer les garanties des solveurs

> **Navigation** : [Partie 5](README.md) · [App-23 PRESENT/SAT](../Part3-Cas-pratiques/App-23-PRESENT-Differential-Cryptanalysis-SAT-Python.ipynb) · [Search-3 A*](../Part1-Foundations/Search-03-Informed.ipynb) · [CSP-4 Scheduling](../Part2-CSP/CSP-4-Scheduling.ipynb) · [Jumeau Python](Frontieres-03-MAPF-Guarantee-Audit-Python.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. valider indépendamment des trajectoires MAPF (mouvements, conflits sommet et arête) ;
2. distinguer **faisabilité**, **optimalité observée** et **garantie algorithmique** ;
3. construire un petit oracle CP-SAT time-expanded pour le makespan ;
4. auditer les hypothèses d'optimalité de CBS, ECBS et OD-A* ;
5. qualifier honnêtement ce qu'un statut `optimal` établit — ou n'établit pas.

**Prérequis** : A*, heuristique admissible, programmation par contraintes, CP-SAT. **Durée estimée** : 60 min.

## Hommage à un travail étudiant

Ce notebook distille le projet G3 *« Coordination de drones par Multi-Agent Path Finding »* de **Matteo Atkinson** et **Paul Witkowski**, EPITA SCIA, Programmation par Contraintes 2026 :

- [répertoire source](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/tree/main/groupe-G3-Coordination_de_drones_par_Multi-Agent_Path_Finding) ;
- [PR #33](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/pull/33) (rendu), [PR #36](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/pull/36) (lancement), [PR #42](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/pull/42) (slides).

Leur appareil combine quatre solveurs — CP-SAT, CBS, ECBS, OD-A* — une API Flask et une visualisation Three.js 3D. Les slides revendiquent des garanties d'optimalité ou de sous-optimalité bornée. CoursIA ne recopie pas ces solveurs : il rejoue trois scénarios, conserve leurs trajectoires sous licence MIT, puis les soumet à un **validateur** et à un **oracle** indépendants. La question n'est donc pas « quel solveur est le plus rapide ? », mais : **quelle preuve autorise chaque étiquette ?**

> **Provenance.** Les données dérivées, le SHA source, le protocole de collecte et les limites sont consignés dans [`data/app24-mapf-audit/SOURCE.md`](data/app24-mapf-audit/SOURCE.md).

## 1. Charger le snapshot frais

Le notebook est autonome : il lit un petit snapshot JSON issu du rerun du code étudiant, sans importer le dépôt PrCon ni ses solveurs.

Le snapshot est un JSON sérialisé : une clé `scenarios` (grille, bâtiments, drones et trajectoires produites par chaque solveur), les clés `source_commit` et `students`, plus un cas `future_goal_case` exploité en section 8.

In [1]:
#r "nuget: Google.OrTools, 9.15.6755"

using System;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Text;
using System.Text.Json;

// Chargement du rerun frais du projet etudiant PrCon G3 : LocateData remonte les
// dossiers parents pour trouver data/app24-mapf-audit/fresh_runs.json (le carnet
// s execute depuis des repertoires differents). Le snapshot fige le commit source
// et les auteurs credits, c est la provenance de l audit.
public static class MapfIo
{
    public static string LocateData()
    {
        var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
        while (dir != null)
        {
            var candidate = Path.Combine(dir.FullName, "data", "app24-mapf-audit", "fresh_runs.json");
            if (File.Exists(candidate)) { return candidate; }
            dir = dir.Parent;
        }
        return Path.Combine(Directory.GetCurrentDirectory(), "data", "app24-mapf-audit", "fresh_runs.json");
    }

    public static JsonDocument Load(string path)
    {
        return JsonDocument.Parse(File.ReadAllText(path, Encoding.UTF8));
    }

    public static int[] ToVector(JsonElement element)
    {
        var values = new List<int>();
        foreach (var item in element.EnumerateArray()) { values.Add(item.GetInt32()); }
        return values.ToArray();
    }

    public static string Key(int[] position)
    {
        return string.Join(",", position.Select(v => v.ToString(CultureInfo.InvariantCulture)));
    }
}

var dataPath = MapfIo.LocateData();
var dataDoc = MapfIo.Load(dataPath);
var data = dataDoc.RootElement;
int scenarioCount = data.GetProperty("scenarios").GetArrayLength();
string sourceCommit = data.GetProperty("source_commit").GetString() ?? "";
var authors = data.GetProperty("students").EnumerateArray().Select(s => s.GetString() ?? "").ToList();
Console.WriteLine($"Snapshot chargé : {scenarioCount} scénarios, commit source {sourceCommit.Substring(0, 10)}");
Console.WriteLine("Auteurs crédités : " + string.Join(", ", authors));

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Google.OrTools, 9.15.6755

Snapshot chargé : 3 scénarios, commit source b5f3f0351d


Auteurs crédités : Matteo Atkinson, Paul Witkowski


**Interprétation.** Le snapshot nomme ses auteurs et son commit source. Cette traçabilité permet de reproduire le même appareil, mais elle ne transforme pas les sorties sérialisées en vérité : toutes les propriétés importantes seront recalculées ci-dessous.

Concrètement, la sortie committée affiche `Snapshot chargé : 3 scénarios, commit source b5f3f0351d` puis `Auteurs crédités : Matteo Atkinson, Paul Witkowski` : trois scénarios et un commit source identifiable, ce qui ancre le rerun dans un état précis du dépôt étudiant.

## 2. Formaliser MAPF : espace, conflits et objectifs

Un agent occupe une position discrète par pas de temps. Il peut attendre ou rejoindre une case voisine. Deux conflits sont interdits :

- **sommet** : deux agents sur la même position au même instant ;
- **arête (swap)** : deux agents échangent leurs positions pendant le même pas.

Après la fin d'une trajectoire, l'agent reste à son but. Le **makespan** est le dernier temps d'arrivée ; le **flowtime** est la somme des temps d'arrivée.

Concrètement, une trajectoire est une liste de positions indexée par le temps ; la méthode `StableArrival`, définie dans la cellule suivante, détecte le premier instant à partir duquel l'agent reste définitivement au but.

In [2]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text.Json;

// Espace d etats MAPF : ScenarioPositions deroule la grille (x,y[,altitude]) en
// excluant les batiments (un batiment d hauteur h bouche les h premieres
// altitudes) ; Neighbors retourne les mouvements valides sur les 2/3 axes plus
// l attente sur place (un agent peut rester immobile).
public static class MapfModel
{
    public static List<int[]> ScenarioPositions(JsonElement scenario)
    {
        var grid = scenario.GetProperty("grid");
        int rows = grid.GetProperty("rows").GetInt32();
        int cols = grid.GetProperty("cols").GetInt32();
        int alts = grid.TryGetProperty("alts", out var altsElement) ? altsElement.GetInt32() : 1;
        var obstacles = new HashSet<string>();
        if (scenario.TryGetProperty("buildings", out var buildings))
        {
            foreach (var building in buildings.EnumerateArray())
            {
                int row = building.GetProperty("row").GetInt32();
                int col = building.GetProperty("col").GetInt32();
                int height = building.GetProperty("height").GetInt32();
                if (alts == 1)
                {
                    obstacles.Add(MapfIo.Key(new[] { row, col }));
                }
                else
                {
                    for (int altitude = 0; altitude < height; altitude++)
                    {
                        obstacles.Add(MapfIo.Key(new[] { row, col, altitude }));
                    }
                }
            }
        }
        var positions = new List<int[]>();
        for (int r = 0; r < rows; r++)
        {
            for (int c = 0; c < cols; c++)
            {
                if (alts == 1)
                {
                    if (!obstacles.Contains(MapfIo.Key(new[] { r, c }))) { positions.Add(new[] { r, c }); }
                }
                else
                {
                    for (int a = 0; a < alts; a++)
                    {
                        if (!obstacles.Contains(MapfIo.Key(new[] { r, c, a }))) { positions.Add(new[] { r, c, a }); }
                    }
                }
            }
        }
        return positions;
    }

    public static List<int[]> Neighbors(int[] position, IEnumerable<int[]> validPositions)
    {
        var valid = new HashSet<string>(validPositions.Select(MapfIo.Key));
        var candidates = new List<int[]> { position };
        for (int axis = 0; axis < position.Length; axis++)
        {
            foreach (int delta in new[] { -1, 1 })
            {
                var moved = (int[])position.Clone();
                moved[axis] += delta;
                candidates.Add(moved);
            }
        }
        return candidates.Where(candidate => valid.Contains(MapfIo.Key(candidate))).ToList();
    }

    public static int StableArrival(IReadOnlyList<int[]> path, int[] goal)
    {
        string goalKey = MapfIo.Key(goal);
        for (int t = 0; t < path.Count; t++)
        {
            bool stable = true;
            for (int i = t; i < path.Count; i++)
            {
                if (MapfIo.Key(path[i]) != goalKey) { stable = false; break; }
            }
            if (stable) { return t; }
        }
        throw new InvalidOperationException("Aucune arrivée stable trouvée pour cette trajectoire");
    }

    public static (int makespan, int flowtime) Objectives(JsonElement drones, JsonElement paths)
    {
        var arrivals = new List<int>();
        foreach (var drone in drones.EnumerateArray())
        {
            string agent = drone.GetProperty("id").GetInt32().ToString(CultureInfo.InvariantCulture);
            var path = paths.GetProperty(agent).EnumerateArray().Select(MapfIo.ToVector).ToList();
            var goal = MapfIo.ToVector(drone.GetProperty("goal"));
            arrivals.Add(StableArrival(path, goal));
        }
        return (arrivals.Max(), arrivals.Sum());
    }
}

foreach (var scenario in data.GetProperty("scenarios").EnumerateArray())
{
    string name = scenario.GetProperty("name").GetString() ?? "";
    int positions = MapfModel.ScenarioPositions(scenario).Count;
    int agents = scenario.GetProperty("drones").GetArrayLength();
    string positionsText = positions.ToString(CultureInfo.InvariantCulture).PadLeft(2);
    string agentsText = agents.ToString(CultureInfo.InvariantCulture);
    Console.WriteLine($"{name,-12} : {positionsText} positions, {agentsText} agents");
}

micro_flat   : 16 positions, 3 agents


micro_3d     : 44 positions, 4 agents


medium_flat  : 48 positions, 5 agents


**Interprétation.** Les trois instances couvrent une grille 2D minimale, une grille 3D avec bâtiments et une grille 2D moyenne. Les objectifs sont recalculés depuis les trajectoires plutôt que crus depuis les champs `makespan` et `flowtime`.

La sortie imprime exactement `micro_flat   : 16 positions, 3 agents`, `micro_3d     : 44 positions, 4 agents` et `medium_flat  : 48 positions, 5 agents` : les positions sont les cases libres de chaque grille (la 3D compte les altitudes), et le nombre d'agents croît avec la taille du scénario.

## 3. Un validateur indépendant répond seulement « faisable ? »

Le validateur ci-dessous ne connaît ni CP-SAT, ni CBS, ni OD-A*. Il vérifie les endpoints, les mouvements, les conflits et la persistance au but.

In [3]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text.Json;

public record ValidationRow(string Scenario, string Solver, string DeclaredStatus, int Makespan, int Flowtime, string Validation);

// Validateur INDEPENDANT du solveur etudiant : chaque trajectoire est relue contre
// la grille (endpoints, positions valides, mouvements voisins), puis chaque pas de
// temps contre les collisions vertex et swap. Le makespan/flowtime recalcules sont
// ceux du rerun, pas ceux declares par le solveur -- c est l ecart mesure qui
public static class MapfValidator
{
    public static List<string> ValidatePaths(JsonElement scenario, JsonElement paths)
    {
        var issues = new List<string>();
        var drones = scenario.GetProperty("drones");
        var positions = MapfModel.ScenarioPositions(scenario);
        var valid = new HashSet<string>(positions.Select(MapfIo.Key));
        var expectedIds = drones.EnumerateArray()
            .Select(d => d.GetProperty("id").GetInt32().ToString(CultureInfo.InvariantCulture))
            .ToHashSet();
        var actualIds = paths.EnumerateObject().Select(p => p.Name).ToHashSet();
        if (!actualIds.SetEquals(expectedIds))
        {
            return new List<string> { "ensemble d'agents incorrect" };
        }

        var trajectories = new Dictionary<string, List<int[]>>();
        foreach (var drone in drones.EnumerateArray())
        {
            string agent = drone.GetProperty("id").GetInt32().ToString(CultureInfo.InvariantCulture);
            var path = paths.GetProperty(agent).EnumerateArray().Select(MapfIo.ToVector).ToList();
            trajectories[agent] = path;
            var start = MapfIo.ToVector(drone.GetProperty("start"));
            var goal = MapfIo.ToVector(drone.GetProperty("goal"));
            if (path.Count == 0 || MapfIo.Key(path[0]) != MapfIo.Key(start) || MapfIo.Key(path[path.Count - 1]) != MapfIo.Key(goal))
            {
                issues.Add($"agent {agent}: endpoints invalides");
            }
            for (int t = 0; t < path.Count; t++)
            {
                if (!valid.Contains(MapfIo.Key(path[t])))
                {
                    issues.Add($"agent {agent}: position invalide à t={t}");
                }
                if (t > 0)
                {
                    bool legal = MapfModel.Neighbors(path[t - 1], positions).Any(q => MapfIo.Key(q) == MapfIo.Key(path[t]));
                    if (!legal)
                    {
                        issues.Add($"agent {agent}: mouvement invalide à t={t}");
                    }
                }
            }
        }

        int horizon = trajectories.Values.Max(p => p.Count);
        var agents = trajectories.Keys.ToList();
        for (int t = 0; t < horizon; t++)
        {
            var now = new Dictionary<string, string>();
            foreach (var agent in agents)
            {
                var path = trajectories[agent];
                now[agent] = MapfIo.Key(path[Math.Min(t, path.Count - 1)]);
            }
            if (now.Values.Distinct().Count() != now.Count)
            {
                issues.Add($"conflit sommet à t={t}");
            }
            if (t + 1 < horizon)
            {
                var next = new Dictionary<string, string>();
                foreach (var agent in agents)
                {
                    var path = trajectories[agent];
                    next[agent] = MapfIo.Key(path[Math.Min(t + 1, path.Count - 1)]);
                }
                for (int i = 0; i < agents.Count; i++)
                {
                    for (int j = i + 1; j < agents.Count; j++)
                    {
                        string first = agents[i];
                        string second = agents[j];
                        if (now[first] == next[second] && now[second] == next[first])
                        {
                            issues.Add($"conflit arête {first}/{second} à t={t}");
                        }
                    }
                }
            }
        }
        return issues;
    }
}

var validationRows = new List<ValidationRow>();
foreach (var scenario in data.GetProperty("scenarios").EnumerateArray())
{
    string scenarioName = scenario.GetProperty("name").GetString() ?? "";
    foreach (var run in scenario.GetProperty("runs").EnumerateArray())
    {
        var computed = MapfModel.Objectives(scenario.GetProperty("drones"), run.GetProperty("paths"));
        var issues = MapfValidator.ValidatePaths(scenario, run.GetProperty("paths"));
        validationRows.Add(new ValidationRow(
            scenarioName,
            run.GetProperty("solver").GetString() ?? "",
            run.GetProperty("declared_status").GetString() ?? "",
            computed.makespan,
            computed.flowtime,
            issues.Count == 0 ? "OK" : string.Join("; ", issues)));
    }
}

string[] headers = { "scénario", "solveur", "statut déclaré", "makespan recalculé", "flowtime recalculé", "validation" };
int w0 = Math.Max(headers[0].Length, validationRows.Max(r => r.Scenario.Length));
int w1 = Math.Max(headers[1].Length, validationRows.Max(r => r.Solver.Length));
int w2 = Math.Max(headers[2].Length, validationRows.Max(r => r.DeclaredStatus.Length));
int w3 = Math.Max(headers[3].Length, validationRows.Max(r => r.Makespan.ToString(CultureInfo.InvariantCulture).Length));
int w4 = Math.Max(headers[4].Length, validationRows.Max(r => r.Flowtime.ToString(CultureInfo.InvariantCulture).Length));
int w5 = Math.Max(headers[5].Length, validationRows.Max(r => r.Validation.Length));

Console.WriteLine($"{headers[0].PadLeft(w0)} {headers[1].PadLeft(w1)} {headers[2].PadLeft(w2)} {headers[3].PadLeft(w3)} {headers[4].PadLeft(w4)} {headers[5].PadLeft(w5)}");
foreach (var row in validationRows)
{
    Console.WriteLine($"{row.Scenario.PadLeft(w0)} {row.Solver.PadLeft(w1)} {row.DeclaredStatus.PadLeft(w2)} {row.Makespan.ToString(CultureInfo.InvariantCulture).PadLeft(w3)} {row.Flowtime.ToString(CultureInfo.InvariantCulture).PadLeft(w4)} {row.Validation.PadLeft(w5)}");
}

int okCount = validationRows.Count(r => r.Validation == "OK");
Console.WriteLine($"Exécutions validées OK : {okCount}/{validationRows.Count}");
if (okCount != validationRows.Count)
{
    throw new InvalidOperationException("Trajectoire invalide détectée par le validateur indépendant");
}

   scénario solveur statut déclaré makespan recalculé flowtime recalculé validation


 micro_flat  CP-SAT        optimal                  6                 18         OK


 micro_flat     CBS        optimal                  6                 18         OK


 micro_flat   OD-A*        optimal                  8                 20         OK


   micro_3d  CP-SAT        optimal                  8                 30         OK


   micro_3d     CBS        optimal                  8                 30         OK


   micro_3d   OD-A*        optimal                  8                 30         OK


medium_flat  CP-SAT        optimal                 12                 54         OK


medium_flat     CBS        optimal                 12                 49         OK


medium_flat   OD-A*        optimal                 16                 60         OK


Exécutions validées OK : 9/9


**Interprétation.** Toutes les trajectoires du snapshot passent les contrôles indépendants. Les temps d'arrivée sont comptés à partir de l'instant où chaque agent reste définitivement au but : une visite suivie d'un départ n'est pas une arrivée. Ce résultat établit la **faisabilité** sur ces trois instances, sans prouver l'optimalité du makespan ou du flowtime.

La sortie committée liste les 9 exécutions, toutes `OK` :

| scénario | solveur | statut déclaré | makespan recalculé | flowtime recalculé |
|---|---|---|---|---|
| micro_flat | CP-SAT | optimal | 6 | 18 |
| micro_flat | CBS | optimal | 6 | 18 |
| micro_flat | OD-A* | optimal | 8 | 20 |
| micro_3d | CP-SAT | optimal | 8 | 30 |
| micro_3d | CBS | optimal | 8 | 30 |
| micro_3d | OD-A* | optimal | 8 | 30 |
| medium_flat | CP-SAT | optimal | 12 | 54 |
| medium_flat | CBS | optimal | 12 | 49 |
| medium_flat | OD-A* | optimal | 16 | 60 |

Dès cette table, OD-A* affiche un makespan supérieur à CP-SAT et CBS sur `micro_flat` (8 contre 6) et `medium_flat` (16 contre 12) malgré le statut `optimal` déclaré — la section 6 confrontera ces écarts à l'oracle.

### Exercice 1 — Détecter un swap

Complétez un contrôle isolé qui reçoit deux chemins et retourne les instants où les agents échangent leurs positions.

**Indice :** comparer `(a[t], a[t+1])` à `(b[t+1], b[t])`.

In [4]:
// Illustration 1 — Exercice 1 : Détecter un swap
using System;
using System.Collections.Generic;

public static List<int> SwapTimes(IReadOnlyList<int[]> pathA, IReadOnlyList<int[]> pathB)
{
    // TODO étudiant : parcourir l'horizon commun et retourner les instants de swap.
    return new List<int>();
}

Console.WriteLine("Exercice a completer");

Exercice a completer


**Point de contrôle.** Un conflit arête est temporel : deux agents peuvent visiter les mêmes cases à des instants différents sans collision.

Concrètement, un swap est un échange simultané : si `(a[t], a[t+1]) == (b[t+1], b[t])`, les deux agents empruntent la même arête en sens inverse au pas `t`, ce que le validateur de la section 3 signale `conflit arête`.

## 4. Makespan et flowtime ne racontent pas la même histoire

Deux solutions peuvent s'inverser selon l'objectif. L'exemple guide suivant utilise directement des temps d'arrivée.

In [5]:
using System;
using System.Globalization;
using System.Linq;

// Deux metriques distinctes : makespan = max des arrives (duree totale de la
// mission), flowtime = somme des arrives (temps cumule des agents). Une solution A
// meilleure en makespan peut etre pire en flowtime -- le choix d optimisation
// n est pas neutre.
var arrivalsA = new[] { 2, 2, 8 };
var arrivalsB = new[] { 5, 5, 5 };
var metrics = new[]
{
    new { Solution = "A", Makespan = arrivalsA.Max(), Flowtime = arrivalsA.Sum() },
    new { Solution = "B", Makespan = arrivalsB.Max(), Flowtime = arrivalsB.Sum() },
};

Console.WriteLine($"{"solution".PadLeft(8)} {"makespan".PadLeft(9)} {"flowtime".PadLeft(9)}");
foreach (var metric in metrics)
{
    Console.WriteLine($"{metric.Solution.PadLeft(8)} {metric.Makespan.ToString(CultureInfo.InvariantCulture).PadLeft(9)} {metric.Flowtime.ToString(CultureInfo.InvariantCulture).PadLeft(9)}");
}
Console.WriteLine("Préférence makespan : " + metrics.OrderBy(m => m.Makespan).First().Solution);
Console.WriteLine("Préférence flowtime : " + metrics.OrderBy(m => m.Flowtime).First().Solution);

solution  makespan  flowtime


       A         8        12


       B         5        15


Préférence makespan : B


Préférence flowtime : A


**Interprétation.** B minimise le makespan, tandis que A minimise le flowtime. Une étiquette `optimal` sans objectif nommé est donc incomplète.

Les valeurs committées sont `A : makespan 8, flowtime 12` et `B : makespan 5, flowtime 15`, d'où les deux lignes `Préférence makespan : B` et `Préférence flowtime : A` : aucune des deux solutions ne domine l'autre sur les deux métriques à la fois.

### Exercice 2 — Construire une inversion d'objectifs

Proposez deux vecteurs de temps d'arrivée pour quatre agents tels que le premier ait le meilleur makespan et le second le meilleur flowtime.

In [6]:
// Illustration 2 — Exercice 2 : Construire une inversion d'objectifs
#nullable enable
using System;

// TODO étudiant : remplacer ces valeurs par deux vecteurs qui inversent les préférences.
int[]? arrivalsFirst = null;
int[]? arrivalsSecond = null;

Console.WriteLine("Exercice a completer");

Exercice a completer


**Point de contrôle.** Avant de comparer des solveurs, écrire noir sur blanc l'objectif qu'ils optimisent évite de transformer une différence de métrique en défaut algorithmique.

Méthode : pour chaque vecteur de quatre temps d'arrivée, calculer `makespan = max(...)` et `flowtime = sum(...)`, puis comparer les `argmin` des deux métriques — le même calcul que la cellule de démonstration ci-dessus, avec quatre agents au lieu de trois.

## 5. Un oracle CP-SAT indépendant pour le makespan

Pour chaque horizon `H`, l'oracle crée les booléens `x[a,p,t]`, impose une position par agent et par temps, les mouvements légaux, la persistance au but, puis interdit conflits sommet et swaps. Le premier horizon faisable est le makespan optimal **du modèle fini**.

In [7]:
using Google.OrTools.Sat;
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text.Json;

// Oracle CP-SAT time-expanded : variable booleenne x[a,p,t] = agent a a la
// position p au temps t. AddExactlyOne impose une seule position par agent et
// temps ; contraintes start/goal au temps 0/horizon, but absorbe (une fois
// atteint on y reste), et non-collision temps par temps. La reference optinale
// independante qui tranche la garantie du solveur etudiant.
public static class MapfOracle
{
    public static int Manhattan(IReadOnlyList<int> first, IReadOnlyList<int> second)
    {
        return first.Zip(second, (a, b) => Math.Abs(a - b)).Sum();
    }

    public static (Dictionary<string, List<int[]>> Paths, string StatusName, bool Found) SolveAtHorizon(
        JsonElement scenario,
        int horizon,
        double timeLimit = 20.0)
    {
        var positions = MapfModel.ScenarioPositions(scenario);
        var index = positions.Select((position, i) => (position, i))
            .ToDictionary(item => MapfIo.Key(item.position), item => item.i);
        var neighbors = positions
            .Select(position => MapfModel.Neighbors(position, positions)
                .Select(neighbor => index[MapfIo.Key(neighbor)])
                .ToList())
            .ToList();
        var drones = scenario.GetProperty("drones").EnumerateArray().ToList();
        var model = new CpModel();
        var x = new Dictionary<(int Agent, int Position, int Time), BoolVar>();

        for (int agent = 0; agent < drones.Count; agent++)
        {
            for (int position = 0; position < positions.Count; position++)
            {
                for (int time = 0; time <= horizon; time++)
                {
                    x[(agent, position, time)] = model.NewBoolVar($"x_{agent}_{position}_{time}");
                }
            }
        }

        for (int agent = 0; agent < drones.Count; agent++)
        {
            var drone = drones[agent];
            for (int time = 0; time <= horizon; time++)
            {
                model.AddExactlyOne(Enumerable.Range(0, positions.Count)
                    .Select(position => (ILiteral)x[(agent, position, time)]));
            }

            int startIndex = index[MapfIo.Key(MapfIo.ToVector(drone.GetProperty("start")))];
            int goalIndex = index[MapfIo.Key(MapfIo.ToVector(drone.GetProperty("goal")))];
            model.Add(x[(agent, startIndex, 0)] == 1);
            model.Add(x[(agent, goalIndex, horizon)] == 1);

            for (int time = 0; time < horizon; time++)
            {
                model.Add(x[(agent, goalIndex, time)] <= x[(agent, goalIndex, time + 1)]);
                for (int position = 0; position < positions.Count; position++)
                {
                    LinearExpr nextPositions = LinearExpr.Sum(neighbors[position]
                        .Select(next => (ILiteral)x[(agent, next, time + 1)]));
                    model.Add(x[(agent, position, time)] <= nextPositions);
                }
            }
        }

        for (int time = 0; time <= horizon; time++)
        {
            for (int position = 0; position < positions.Count; position++)
            {
                model.AddAtMostOne(Enumerable.Range(0, drones.Count)
                    .Select(agent => (ILiteral)x[(agent, position, time)]));
            }
        }

        var directedEdges = new List<(int From, int To)>();
        for (int from = 0; from < positions.Count; from++)
        {
            foreach (int to in neighbors[from])
            {
                if (from != to) { directedEdges.Add((from, to)); }
            }
        }

        for (int time = 0; time < horizon; time++)
        {
            foreach (var edge in directedEdges)
            {
                for (int first = 0; first < drones.Count; first++)
                {
                    for (int second = first + 1; second < drones.Count; second++)
                    {
                        model.Add(
                            x[(first, edge.From, time)] + x[(first, edge.To, time + 1)]
                            + x[(second, edge.To, time)] + x[(second, edge.From, time + 1)] <= 3);
                    }
                }
            }
        }

        var solver = new CpSolver
        {
            StringParameters = $"max_time_in_seconds={timeLimit.ToString(CultureInfo.InvariantCulture)} num_search_workers=4"
        };
        solver.Solve(model);
        CpSolverStatus status = solver.Response.Status;
        string statusName = status.ToString();
        if (status != CpSolverStatus.Optimal && status != CpSolverStatus.Feasible)
        {
            return (new Dictionary<string, List<int[]>>(), statusName, false);
        }

        var paths = new Dictionary<string, List<int[]>>();
        for (int agent = 0; agent < drones.Count; agent++)
        {
            string agentId = drones[agent].GetProperty("id").GetInt32()
                .ToString(CultureInfo.InvariantCulture);
            var path = new List<int[]>();
            for (int time = 0; time <= horizon; time++)
            {
                int position = Enumerable.Range(0, positions.Count)
                    .Single(candidate => solver.BooleanValue(x[(agent, candidate, time)]));
                path.Add(positions[position]);
            }
            paths[agentId] = path;
        }
        return (paths, statusName, true);
    }

    public static (int Optimum, Dictionary<string, List<int[]>> Paths, string StatusName, int Lower) MakespanOracle(
        JsonElement scenario)
    {
        var drones = scenario.GetProperty("drones").EnumerateArray().ToList();
        int lower = drones.Max(drone => Manhattan(
            MapfIo.ToVector(drone.GetProperty("start")),
            MapfIo.ToVector(drone.GetProperty("goal"))));

        for (int horizon = lower; horizon < lower + 21; horizon++)
        {
            var result = SolveAtHorizon(scenario, horizon);
            if (result.Found)
            {
                return (horizon, result.Paths, result.StatusName, lower);
            }
        }
        throw new InvalidOperationException("Aucune solution trouvée dans la fenêtre d'horizons auditée");
    }
}

Console.WriteLine("Oracle défini : recherche croissante du premier horizon faisable");

Oracle défini : recherche croissante du premier horizon faisable


**Interprétation.** L'oracle ne réutilise ni l'horizon ni les contraintes du solveur étudiant. La recherche croissante rend la borne inférieure et le premier horizon faisable visibles ; le statut CP-SAT reste attaché à chaque modèle fini.

La sortie se limite au print `Oracle défini : recherche croissante du premier horizon faisable` : à ce stade aucune résolution n'a encore tourné ; la borne inférieure de départ sera le maximum des distances de Manhattan start→goal, et l'oracle fera monter l'horizon jusqu'au premier modèle faisable.

In [8]:
using Google.OrTools.Sat;
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text.Json;

public record OracleRow(string Scenario, int LowerBound, int OracleMakespan, string CpSatStatus, string Validation);

var oracleRows = new List<OracleRow>();
var oraclePaths = new Dictionary<string, Dictionary<string, List<int[]>>>();

// Résolution CP-SAT à horizon fixé : premier horizon faisable = optimum du modèle fini.
(Dictionary<string, List<int[]>> paths, string statusName, bool found) SolveAtHorizon(JsonElement scenario, int horizon, double timeLimit)
{
    var positions = MapfModel.ScenarioPositions(scenario);
    var index = positions.Select((position, i) => (position, i))
        .ToDictionary(item => MapfIo.Key(item.position), item => item.i);
    var neighbors = positions
        .Select(position => MapfModel.Neighbors(position, positions)
            .Select(neighbor => index[MapfIo.Key(neighbor)])
            .ToList())
        .ToList();
    var drones = scenario.GetProperty("drones").EnumerateArray().ToList();
    var model = new CpModel();
    var x = new Dictionary<(int Agent, int Position, int Time), BoolVar>();

    for (int agent = 0; agent < drones.Count; agent++)
    {
        for (int position = 0; position < positions.Count; position++)
        {
            for (int time = 0; time <= horizon; time++)
            {
                x[(agent, position, time)] = model.NewBoolVar($"x_{agent}_{position}_{time}");
            }
        }
    }

    for (int agent = 0; agent < drones.Count; agent++)
    {
        var drone = drones[agent];
        for (int time = 0; time <= horizon; time++)
        {
            model.AddExactlyOne(Enumerable.Range(0, positions.Count)
                .Select(position => (ILiteral)x[(agent, position, time)]));
        }

        int startIndex = index[MapfIo.Key(MapfIo.ToVector(drone.GetProperty("start")))];
        int goalIndex = index[MapfIo.Key(MapfIo.ToVector(drone.GetProperty("goal")))];
        model.Add(x[(agent, startIndex, 0)] == 1);
        model.Add(x[(agent, goalIndex, horizon)] == 1);

        for (int time = 0; time < horizon; time++)
        {
            model.Add(x[(agent, goalIndex, time)] <= x[(agent, goalIndex, time + 1)]);
            for (int position = 0; position < positions.Count; position++)
            {
                LinearExpr nextPositions = LinearExpr.Sum(neighbors[position]
                    .Select(next => (ILiteral)x[(agent, next, time + 1)]));
                model.Add(x[(agent, position, time)] <= nextPositions);
            }
        }
    }

    for (int time = 0; time <= horizon; time++)
    {
        for (int position = 0; position < positions.Count; position++)
        {
            model.AddAtMostOne(Enumerable.Range(0, drones.Count)
                .Select(agent => (ILiteral)x[(agent, position, time)]));
        }
    }

    var directedEdges = new List<(int From, int To)>();
    for (int from = 0; from < positions.Count; from++)
    {
        foreach (int to in neighbors[from])
        {
            if (from != to) { directedEdges.Add((from, to)); }
        }
    }

    for (int time = 0; time < horizon; time++)
    {
        foreach (var edge in directedEdges)
        {
            for (int first = 0; first < drones.Count; first++)
            {
                for (int second = first + 1; second < drones.Count; second++)
                {
                    model.Add(
                        x[(first, edge.From, time)] + x[(first, edge.To, time + 1)]
                        + x[(second, edge.To, time)] + x[(second, edge.From, time + 1)] <= 3);
                }
            }
        }
    }

    var solver = new CpSolver
    {
        StringParameters = $"max_time_in_seconds:{timeLimit.ToString(CultureInfo.InvariantCulture)} num_search_workers:4"
    };
    solver.Solve(model);
    CpSolverStatus status = solver.Response.Status;
    string statusName = status.ToString().ToUpperInvariant();
    if (status != CpSolverStatus.Optimal && status != CpSolverStatus.Feasible)
    {
        return (new Dictionary<string, List<int[]>>(), statusName, false);
    }

    var solved = new Dictionary<string, List<int[]>>();
    for (int agent = 0; agent < drones.Count; agent++)
    {
        string agentId = drones[agent].GetProperty("id").GetInt32()
            .ToString(CultureInfo.InvariantCulture);
        var path = new List<int[]>();
        for (int time = 0; time <= horizon; time++)
        {
            int position = Enumerable.Range(0, positions.Count)
                .Single(candidate => solver.BooleanValue(x[(agent, candidate, time)]));
            path.Add(positions[position]);
        }
        solved[agentId] = path;
    }
    return (solved, statusName, true);
}

foreach (var scenario in data.GetProperty("scenarios").EnumerateArray())
{
    string scenarioName = scenario.GetProperty("name").GetString() ?? "";
    var drones = scenario.GetProperty("drones").EnumerateArray().ToList();
    int lower = drones.Max(drone => MapfIo.ToVector(drone.GetProperty("start"))
        .Zip(MapfIo.ToVector(drone.GetProperty("goal")), (a, b) => Math.Abs(a - b))
        .Sum());

    for (int horizon = lower; horizon < lower + 21; horizon++)
    {
        var result = SolveAtHorizon(scenario, horizon, 20.0);
        if (!result.found) { continue; }

        var serializedPaths = JsonSerializer.SerializeToElement(result.paths);
        var issues = MapfValidator.ValidatePaths(scenario, serializedPaths);
        if (issues.Count != 0)
        {
            throw new InvalidOperationException("Trajectoire oracle invalide : " + string.Join("; ", issues));
        }

        oraclePaths[scenarioName] = result.paths;
        oracleRows.Add(new OracleRow(scenarioName, lower, horizon, result.statusName, "OK"));
        break;
    }
    if (oracleRows.Count(o => o.Scenario == scenarioName) == 0)
    {
        throw new InvalidOperationException("Aucun horizon faisable pour le scénario " + scenarioName);
    }
}

string[] headers = { "scénario", "borne inférieure", "makespan oracle", "statut CP-SAT", "validation" };
int w0 = Math.Max(headers[0].Length, oracleRows.Max(r => r.Scenario.Length));
int w1 = Math.Max(headers[1].Length, oracleRows.Max(r => r.LowerBound.ToString(CultureInfo.InvariantCulture).Length));
int w2 = Math.Max(headers[2].Length, oracleRows.Max(r => r.OracleMakespan.ToString(CultureInfo.InvariantCulture).Length));
int w3 = Math.Max(headers[3].Length, oracleRows.Max(r => r.CpSatStatus.Length));
int w4 = Math.Max(headers[4].Length, oracleRows.Max(r => r.Validation.Length));

Console.WriteLine($"{headers[0].PadLeft(w0)} {headers[1].PadLeft(w1)} {headers[2].PadLeft(w2)} {headers[3].PadLeft(w3)} {headers[4].PadLeft(w4)}");
foreach (var row in oracleRows)
{
    Console.WriteLine($"{row.Scenario.PadLeft(w0)} {row.LowerBound.ToString(CultureInfo.InvariantCulture).PadLeft(w1)} {row.OracleMakespan.ToString(CultureInfo.InvariantCulture).PadLeft(w2)} {row.CpSatStatus.PadLeft(w3)} {row.Validation.PadLeft(w4)}");
}

   scénario borne inférieure makespan oracle statut CP-SAT validation


 micro_flat                6               6       OPTIMAL         OK


   micro_3d                8               8       OPTIMAL         OK


medium_flat               12              12       OPTIMAL         OK


**Interprétation.** Sur chaque petite instance, la borne de distance et le premier horizon faisable coïncident ou sont séparés par la coordination requise. Le résultat établit un makespan minimal pour ce modèle explicite et cette recherche d'horizons — pas une garantie universelle sur le code étudiant.

La sortie affiche pour chaque scénario `borne inférieure = makespan oracle` avec le statut `OPTIMAL` : `micro_flat` 6/6, `micro_3d` 8/8, `medium_flat` 12/12. La borne de distance est donc atteinte telle quelle sur ces trois instances, et chaque trajectoire oracle repasse le validateur (`validation OK`).

## 6. Réfuter un statut `optimal` par un contre-exemple

Comparons maintenant les makespans déclarés à l'oracle indépendant.

In [9]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Croisement des makespans recalcules par le validateur avec ceux de l oracle :
// la colonne MakespanGap > 0 marque une refutation de la sous-optimalite
// declaree. OD-A* est cense optimal -- tout ecart est une preuve de defaut.
public record ComparisonRow(string Scenario, string Solver, string DeclaredStatus, int RecomputedMakespan, int OracleMakespan, int MakespanGap);

var oracleByScenario = oracleRows.ToDictionary(r => r.Scenario, r => r.OracleMakespan);
var comparison = new List<ComparisonRow>();
foreach (var row in validationRows)
{
    int oracleMakespan = oracleByScenario[row.Scenario];
    comparison.Add(new ComparisonRow(row.Scenario, row.Solver, row.DeclaredStatus, row.Makespan, oracleMakespan, row.Makespan - oracleMakespan));
}

string[] headers = { "scénario", "solveur", "statut déclaré", "makespan recalculé", "makespan oracle", "écart makespan" };
int w0 = Math.Max(headers[0].Length, comparison.Max(r => r.Scenario.Length));
int w1 = Math.Max(headers[1].Length, comparison.Max(r => r.Solver.Length));
int w2 = Math.Max(headers[2].Length, comparison.Max(r => r.DeclaredStatus.Length));
int w3 = Math.Max(headers[3].Length, comparison.Max(r => r.RecomputedMakespan.ToString(CultureInfo.InvariantCulture).Length));
int w4 = Math.Max(headers[4].Length, comparison.Max(r => r.OracleMakespan.ToString(CultureInfo.InvariantCulture).Length));
int w5 = Math.Max(headers[5].Length, comparison.Max(r => r.MakespanGap.ToString(CultureInfo.InvariantCulture).Length));

Console.WriteLine($"{headers[0].PadLeft(w0)} {headers[1].PadLeft(w1)} {headers[2].PadLeft(w2)} {headers[3].PadLeft(w3)} {headers[4].PadLeft(w4)} {headers[5].PadLeft(w5)}");
foreach (var row in comparison)
{
    Console.WriteLine($"{row.Scenario.PadLeft(w0)} {row.Solver.PadLeft(w1)} {row.DeclaredStatus.PadLeft(w2)} {row.RecomputedMakespan.ToString(CultureInfo.InvariantCulture).PadLeft(w3)} {row.OracleMakespan.ToString(CultureInfo.InvariantCulture).PadLeft(w4)} {row.MakespanGap.ToString(CultureInfo.InvariantCulture).PadLeft(w5)}");
}

Console.WriteLine();
Console.WriteLine("Histogramme des makespans (une barre = un pas de temps) :");
foreach (var group in comparison.GroupBy(r => r.Scenario))
{
    Console.WriteLine(group.Key);
    foreach (var row in group)
    {
        string bar = new string('#', row.RecomputedMakespan);
        Console.WriteLine($"  {row.Solver,-8} recalculé {bar} ({row.RecomputedMakespan})");
    }
    int oracleValue = oracleByScenario[group.Key];
    Console.WriteLine($"  {"Oracle",-8} oracle    {new string('#', oracleValue)} ({oracleValue})");
}

var odRefutations = comparison.Where(r => r.Solver == "OD-A*" && r.MakespanGap > 0).ToList();
Console.WriteLine();
Console.WriteLine($"Contre-exemples OD-A* (makespan recalculé > makespan oracle) : {odRefutations.Count}");
foreach (var row in odRefutations)
{
    Console.WriteLine($"  {row.Scenario} : recalculé {row.RecomputedMakespan}, oracle {row.OracleMakespan}, écart {row.MakespanGap}");
}
if (odRefutations.Count != 2)
{
    throw new InvalidOperationException($"Nombre de contre-exemples OD-A* inattendu : {odRefutations.Count} (2 attendu)");
}

   scénario solveur statut déclaré makespan recalculé makespan oracle écart makespan


 micro_flat  CP-SAT        optimal                  6               6              0


 micro_flat     CBS        optimal                  6               6              0


 micro_flat   OD-A*        optimal                  8               6              2


   micro_3d  CP-SAT        optimal                  8               8              0


   micro_3d     CBS        optimal                  8               8              0


... (19 outputs left out)

  Oracle   oracle    ############ (12)


Contre-exemples OD-A* (makespan recalculé > makespan oracle) : 2


  micro_flat : recalculé 8, oracle 6, écart 2


  medium_flat : recalculé 16, oracle 12, écart 4


**Interprétation.** OD-A* retourne `optimal` alors que son makespan dépasse l'oracle sur **deux scénarios**. Les trajectoires restent valides : le contre-exemple réfute précisément le **statut d'optimalité**, pas la faisabilité du solveur. Sur le scénario 3D, le makespan est égal mais le flowtime diffère ; cet écart ne réfute pas une prétention limitée au makespan.

Dans le tableau committé, seuls deux écarts sont non nuls : OD-A* sur `micro_flat` (`makespan recalculé 8`, `makespan oracle 6`, `écart makespan 2`) et sur `medium_flat` (`16` contre `12`, `écart makespan 4`) ; CP-SAT et CBS sont à écart 0 partout. C'est exactement ce que le contrôle `odRefutations.Count != 2` de la cellule vérifie.

## 7. Pourquoi l'heuristique OD-A* ne porte pas la garantie annoncée

Dans l'implémentation étudiante, `g` augmente d'une unité par timestep joint, tandis que `h` additionne les distances restantes de tous les agents. Pour minimiser le makespan, le coût restant est gouverné par un **maximum**, pas par une somme.

In [10]:
using System;
using System.Globalization;
using System.Linq;
using System.Text.Json;

// Admissibilite : l heuristique somme des distances de Manhattan est un
// SUR-cout quand les agents bougent en parallele (le makespan reel est au
// moins le max, pas la somme). La garde ci-dessous prouve l inadmissibilite
// sur le micro-scenario : h_sum > borne parallele.
var micro = data.GetProperty("scenarios").EnumerateArray().First();
var remainingDistances = micro.GetProperty("drones").EnumerateArray()
    .Select(drone => MapfIo.ToVector(drone.GetProperty("start"))
        .Zip(MapfIo.ToVector(drone.GetProperty("goal")), (a, b) => Math.Abs(a - b))
        .Sum())
    .ToList();
int hSum = remainingDistances.Sum();
int trueParallelLowerBound = remainingDistances.Max();

Console.WriteLine("Distances individuelles : [" + string.Join(", ", remainingDistances.Select(v => v.ToString(CultureInfo.InvariantCulture))) + "]");
Console.WriteLine("Heuristique somme utilisée : " + hSum.ToString(CultureInfo.InvariantCulture));
Console.WriteLine("Borne makespan par mouvements parallèles : " + trueParallelLowerBound.ToString(CultureInfo.InvariantCulture));
Console.WriteLine("Admissible pour le makespan : " + (hSum <= trueParallelLowerBound));

if (!(hSum > trueParallelLowerBound))
{
    throw new InvalidOperationException("La somme des distances devrait dépasser la borne parallèle sur ce scénario");
}

Distances individuelles : [6, 6, 6]


Heuristique somme utilisée : 18


Borne makespan par mouvements parallèles : 6


Admissible pour le makespan : False


**Interprétation.** Dès l'état initial, la somme surestime le nombre minimal de timesteps parallèles. Elle peut être pertinente pour une somme de coûts, mais elle n'est pas admissible pour le makespan avec ce `g`. Le théorème d'optimalité de A* ne s'applique donc pas à ce couple coût/heuristique.

La sortie donne `Distances individuelles : [6, 6, 6]`, `Heuristique somme utilisée : 18`, `Borne makespan par mouvements parallèles : 6` et conclut `Admissible pour le makespan : False` : la somme des distances (18) dépasse de 12 la borne parallèle (6), puisque les trois agents peuvent avancer en même temps.

## 8. CBS : arriver au but ne suffit pas face à une contrainte future

Le snapshot contient un cas 1×3 : l'agent atteint son but à `t=2`, mais ce même but est interdit à `t=4`. Avec persistance au but, retourner immédiatement à `pos == goal` viole la contrainte future.

In [11]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text.Json;

// CBS : arret au but SOUS contrainte future. Arreter des que pos == goal est
// invalide si une contrainte interdit le but a un temps ulterieur ; le chemin
// low-level doit attendre la fin de la contrainte avant de s y figer. La garde
// ci-dessous verifie que le chemin etudiant echoue ce test.
public static class MapfGoal
{
    public static bool SafeGoalTermination(
        IReadOnlyList<int[]> path,
        int[] goalPosition,
        IEnumerable<(string Position, int Time)> constraints)
    {
        int arrival = path.Count - 1;
        bool atGoal = MapfIo.Key(path[path.Count - 1]) == MapfIo.Key(goalPosition);
        bool futureSafe = constraints.All(constraint =>
            constraint.Position != MapfIo.Key(goalPosition) || constraint.Time <= arrival);
        return atGoal && futureSafe;
    }
}

var futureCase = data.GetProperty("future_goal_case");
var path = futureCase.GetProperty("student_low_level_path").EnumerateArray()
    .Select(MapfIo.ToVector).ToList();
var goal = MapfIo.ToVector(futureCase.GetProperty("goal"));
int forbiddenTime = futureCase.GetProperty("forbidden").GetProperty("time").GetInt32();
var forbiddenPosition = MapfIo.ToVector(futureCase.GetProperty("forbidden").GetProperty("position"));
var constraints = new List<(string Position, int Time)>
{
    (MapfIo.Key(forbiddenPosition), forbiddenTime),
};

string pathText = "[" + string.Join(", ", path.Select(p => "(" + string.Join(", ", p.Select(v => v.ToString(CultureInfo.InvariantCulture))) + ")")) + "]";
bool naiveStop = MapfIo.Key(path[path.Count - 1]) == MapfIo.Key(goal);
bool safeStop = MapfGoal.SafeGoalTermination(path, goal, constraints);
bool constraintRespected = futureCase.GetProperty("constraint_respected").GetBoolean();

Console.WriteLine("Chemin low-level retourné : " + pathText);
Console.WriteLine("Arrêt naïf `pos == goal` : " + naiveStop);
Console.WriteLine("Arrêt sûr avec contraintes futures : " + safeStop);
Console.WriteLine("Contrainte future respectée selon le rerun : " + constraintRespected);

if (!(naiveStop && !safeStop))
{
    throw new InvalidOperationException("Le micro-cas devrait montrer un arrêt naïf sûr et un arrêt sûr refusant la contrainte future");
}

Chemin low-level retourné : [(0, 0), (0, 1), (0, 2)]


Arrêt naïf `pos == goal` : True


Arrêt sûr avec contraintes futures : False


Contrainte future respectée selon le rerun : False


**Interprétation.** Le micro-cas reproduit un défaut du low-level A* : la solution est rendue avant d'examiner une interdiction future au but. Comme CBS et ECBS réutilisent ce low-level et un horizon empirique, les garanties de leurs versions canoniques ne sont pas automatiquement transférées à cette implémentation.

La sortie committée affiche `Chemin low-level retourné : [(0, 0), (0, 1), (0, 2)]`, `Arrêt naïf `pos == goal` : True`, `Arrêt sûr avec contraintes futures : False` et `Contrainte future respectée selon le rerun : False` : le but est atteint à t=2, mais la contrainte l'interdit à t=4.

### Exercice 3 — Restaurer le contrat de garantie

Complétez un prédicat d'arrêt qui n'accepte le but que si aucune contrainte future ne l'interdit. Puis indiquez quelles autres hypothèses (horizon, coût, heuristique) devraient être auditées pour CBS ou ECBS.

**Indice :** chercher le plus grand temps interdit au but.

In [12]:
// Illustration 3 — Exercice 3 : Restaurer le contrat de garantie
#nullable enable
using System;
using System.Collections.Generic;

public static bool? GoalIsSafe(int[] goalPosition, int arrivalTime, IEnumerable<(string Position, int Time)> constraints)
{
    // TODO étudiant : refuser si le but est interdit après arrival_time.
    return null;
}

Console.WriteLine("Exercice a completer");

Exercice a completer


**Point de contrôle.** Réparer le prédicat d'arrêt est nécessaire mais ne prouve pas à lui seul toute l'implémentation : l'horizon et l'ordre de coût restent à justifier.

Au-delà du prédicat d'arrêt, trois hypothèses restent à auditer pour CBS/ECBS : l'horizon de recherche (fini et suffisant ?), la fonction de coût (makespan ou somme des coûts ?), et l'admissibilité du low-level — la section 7 traite cette dernière.

## 9. ECBS : une formule FOCAL n'est pas une preuve complète

ECBS sélectionne classiquement dans un ensemble `FOCAL` borné par `w × f_min`. Cette règle peut soutenir une garantie w-sous-optimale si les bornes, les coûts et le low-level satisfont les hypothèses du théorème. Ici, le low-level présente le défaut reproduit ci-dessus et l'horizon est empirique. Les sorties peuvent être bonnes ; la borne `≤ w × optimum` de **cette implémentation** reste **non établie** par les trois scénarios.

## 10. Matrice des niveaux de preuve

In [13]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Tableau de preuve final : par solveur, ce que le rerun MONTRE, ce que l oracle
// ETABLIT, et ce qui reste NON PROUVE. La distillation honnete du projet
// PrCon G3 -- rien de plus, rien de moins que ce qui est mesure.
public record EvidenceRow(string Solver, string WhatTheRerunShows, string WhatTheOracleEstablishes, string WhatRemainsUnproven);

int odGapCount = comparison.Count(r => r.Solver == "OD-A*" && r.MakespanGap > 0);
var odGaps = comparison
    .Where(r => r.Solver == "OD-A*" && r.MakespanGap > 0)
    .Select(r => r.MakespanGap.ToString(CultureInfo.InvariantCulture))
    .ToList();

var evidence = new List<EvidenceRow>
{
    new("CP-SAT étudiant", "trajectoires valides sur 3 scénarios", "optimum CoursIA sous modèle fini", "suffisance générale de l'horizon étudiant"),
    new("CBS étudiant", "valide et égal à l'oracle sur 3 scénarios", "égalité sur ces instances", "garantie générale (but futur + horizon)"),
    new("ECBS étudiant", "architecture et résultats rapportés", "aucune borne w certifiée ici", "garantie w de cette implémentation"),
    new("OD-A* étudiant", "trajectoires valides", $"sous-optimalité sur {odGapCount.ToString(CultureInfo.InvariantCulture)} scénarios", "statut `optimal`"),
};

string[] headers = { "Solveur", "Ce que montre le rerun", "Ce que l'oracle établit", "Ce qui reste non prouvé" };
int w0 = Math.Max(headers[0].Length, evidence.Max(r => r.Solver.Length));
int w1 = Math.Max(headers[1].Length, evidence.Max(r => r.WhatTheRerunShows.Length));
int w2 = Math.Max(headers[2].Length, evidence.Max(r => r.WhatTheOracleEstablishes.Length));
int w3 = Math.Max(headers[3].Length, evidence.Max(r => r.WhatRemainsUnproven.Length));

Console.WriteLine($"{headers[0].PadRight(w0)} | {headers[1].PadRight(w1)} | {headers[2].PadRight(w2)} | {headers[3].PadRight(w3)}");
Console.WriteLine($"{new string('-', w0)}-+-{new string('-', w1)}-+-{new string('-', w2)}-+-{new string('-', w3)}");
foreach (var row in evidence)
{
    Console.WriteLine($"{row.Solver.PadRight(w0)} | {row.WhatTheRerunShows.PadRight(w1)} | {row.WhatTheOracleEstablishes.PadRight(w2)} | {row.WhatRemainsUnproven.PadRight(w3)}");
}
Console.WriteLine();
Console.WriteLine("Écarts OD-A* mesurés en section 6 : " + string.Join(", ", odGaps));


Solveur         | Ce que montre le rerun                    | Ce que l'oracle établit          | Ce qui reste non prouvé                  


----------------+-------------------------------------------+----------------------------------+------------------------------------------


CP-SAT étudiant | trajectoires valides sur 3 scénarios      | optimum CoursIA sous modèle fini | suffisance générale de l'horizon étudiant


CBS étudiant    | valide et égal à l'oracle sur 3 scénarios | égalité sur ces instances        | garantie générale (but futur + horizon)  


ECBS étudiant   | architecture et résultats rapportés       | aucune borne w certifiée ici     | garantie w de cette implémentation       


OD-A* étudiant  | trajectoires valides                      | sous-optimalité sur 2 scénarios  | statut `optimal`                         


Écarts OD-A* mesurés en section 6 : 2, 4


**Interprétation.** La matrice empêche le glissement le plus fréquent : passer de « le résultat est bon ici » à « l'algorithme est garanti ». Un validateur, un oracle borné et un théorème répondent à trois questions différentes.

## 11. Limites honnêtes

1. **Trois scénarios seulement.** Ils suffisent à produire des contre-exemples, pas à caractériser la scalabilité.
2. **Snapshot dérivé d'un commit.** Une modification future du projet étudiant exige une nouvelle collecte et une nouvelle attribution.
3. **Oracle borné.** La recherche croissante établit le premier horizon faisable des petits modèles construits ici ; elle ne certifie pas toutes les variantes MAPF.
4. **Makespan privilégié.** Le notebook calcule aussi le flowtime mais l'oracle minimise le makespan. Une optimalité flowtime demanderait un autre objectif.
5. **Pas de benchmark runtime.** Les temps absolus dépendent de la machine et ne sont pas utilisés pour classer les solveurs.
6. **Pas de certification de l'application complète.** L'API Flask, la visualisation Three.js, les zones dynamiques et les grands scénarios ne sont pas audités ici.
7. **Pas de preuve formelle.** CP-SAT fournit un statut sur un modèle exécutable ; aucun certificat indépendant ni preuve Lean de l'encodage n'est produit.

## 12. Conclusion

Le geste de Matteo Atkinson et Paul Witkowski est conservé et élargi : leur comparaison de solveurs devient un laboratoire d'**audit des garanties**. Les trajectoires fraîches sont faisables ; l'oracle confirme plusieurs optima observés, réfute deux statuts OD-A*, et le micro-cas CBS révèle une hypothèse d'arrêt violée.

Quatre règles se généralisent au-delà de MAPF :

1. **valide** ne signifie pas **optimal** ;
2. égaler un oracle sur quelques instances ne donne pas une garantie générale ;
3. un statut retourné n'est pas un certificat ;
4. la preuve d'un algorithme canonique ne couvre une implémentation que si ses hypothèses sont préservées.

### Pour aller plus loin

- **Navigation — même famille, autre garantie** : [Frontieres-05-CoveringArrays-Guarantee-Audit](Frontieres-05-CoveringArrays-Guarantee-Audit-Python.ipynb) — séparer solution valide, optimum observé et garantie prouvée sur les tableaux de couverture
- **Partie** : [Index de la Partie 5 — Frontières](README.md)

## Sources

- Stern, R., et al. (2019), *Multi-Agent Pathfinding: Definitions, Variants, and Benchmarks*, SoCS.
- Sharon, G., et al. (2015), *Conflict-Based Search for Optimal Multi-Agent Path Finding*, Artificial Intelligence 219.
- Standley, T. (2010), *Finding Optimal Solutions to Cooperative Pathfinding Problems*, AAAI.
- Barer, M., et al. (2014), *Suboptimal Variants of the Conflict-Based Search Algorithm for the Multi-Agent Pathfinding Problem*, SoCS.
- Projet G3 et historique PrCon : PRs #33, #36 et #42, liens en introduction.

La sortie imprime les quatre lignes du tableau de preuves — `CP-SAT étudiant`, `CBS étudiant`, `ECBS étudiant`, `OD-A* étudiant` — avec pour OD-A* « Ce que l'oracle établit » à `sous-optimalité sur 2 scénarios` et « Ce qui reste non prouvé » à `statut `optimal``, les écarts 2 et 4 mesurés en section 6.

> Carnet issu de l'EPIC #26070 (parité C# de la série Search/Part5-Frontières), conventions de livraison #26534.